In [ ]:
import os
import pickle
import sys
import pandas as pd

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
from src.core.Context import VectorContext
from src.represent.RepresentedVector import RepresentedVector

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

In [3]:
# Semantic pretrained
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

# Topic model đã train
topic_model = BERTopic.load(project_root + "/models/primary/bertopic")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
context = VectorContext(
    os.path.join(project_root, "data", "primary", "dev_set")
)
title_list = context.createTitleList()
titles = [item["title"] for item in title_list]

semantic_embeddings = semantic_model.encode(
    titles,
    show_progress_bar=True,
)

topics, topic_distributions = topic_model.transform(
    titles,
    embeddings=semantic_embeddings,
)

represented_vector_list = {
    item["news_id"]: {
        "semantic": semantic_embedding,
        "topic_distribution": topic_distribution,
    }
    for item, semantic_embedding, topic_distribution in zip(
        title_list,
        semantic_embeddings,
        topic_distributions,
    )
}

len(represented_vector_list)

Batches:   0%|          | 0/2251 [00:00<?, ?it/s]

72023

In [5]:
vectors_dir = os.path.join(project_root, "vectors", "primary")
os.makedirs(vectors_dir, exist_ok=True)

represent_vectors_path = os.path.join(vectors_dir, "represent_vectors.pkl")
with open(represent_vectors_path, "wb") as f:
    pickle.dump(represented_vector_list, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved {len(represented_vector_list)} news vectors to {represent_vectors_path}")

Saved 72023 news vectors to d:\CDNC\MIND-research\vectors\primary\represent_vectors.pkl


In [ ]:
next(iter(represented_vector_list.items()))